# NB03 — RNA-seq and microarray preprocessing
Platform-appropriate preprocessing, never pooling matrices across platforms:
* **Affymetrix arrays (GSE4290, GSE50161):** values range check -> log2 if needed -> quantile normalisation ->
  probe-to-gene mapping with `hgu133plus2.db`, keeping the probe with the highest mean per gene.
* **TCGA + GTEx (Toil):** log2(expected_count + 1) is converted back to expected counts, Ensembl IDs mapped to
  symbols (duplicates summed), low-expression genes filtered by CPM. Counts are kept non-integer (accepted by voom).
* **CGGA (RSEM FPKM):** log2(FPKM + 1). These are *not* counts and are only used for scoring and survival, never for
  count-based differential expression.

Outputs: `<cohort>_*.csv.gz` matrices (genes x samples) in `03_Metadata/matrices/`, PCA and distribution QC figures,
`qc_flags.csv` (outliers are **flagged, never removed**), and a patient-duplicate screen between CGGA batches.

In [27]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import matplotlib
import matplotlib.pyplot as plt
RAW = P("raw"); MAT = P("meta", "matrices"); PRE = CFG["preprocessing"]
COL = {"GBM": "#1f3a93", "control": "#0f9d94", "other": "#999999"}
qc_flags = []
print("SMOKE =", SMOKE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SMOKE = False


In [28]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/GBM_MultiCohort_Research"
)

META = ROOT / "03_Metadata"
MATRICES = META / "matrices"

print("=== NB02 METADATA VERIFICATION ===")

for name in [
    "TCGA_samples.csv",
    "GTEx_samples.csv",
    "CGGA325_samples.csv",
    "CGGA693_samples.csv",
    "cohort_flow.csv",
    "exclusion_log.csv"
]:
    path = META / name

    if path.exists():
        df = pd.read_csv(path)
        print(f"{name}: {len(df)} rows")
    else:
        print(f"MISSING: {name}")

print("\n=== NB03 EXPRESSION MATRICES ===")

if MATRICES.exists():
    files = sorted(MATRICES.glob("*"))

    if files:
        for f in files:
            print(
                f.name,
                "|",
                round(f.stat().st_size / 1024**2, 2),
                "MB"
            )
    else:
        print("Matrix folder exists but is empty.")
else:
    print("Matrix folder does not exist.")

=== NB02 METADATA VERIFICATION ===
TCGA_samples.csv: 153 rows
GTEx_samples.csv: 290 rows
CGGA325_samples.csv: 325 rows
CGGA693_samples.csv: 693 rows
cohort_flow.csv: 4 rows
exclusion_log.csv: 1175 rows

=== NB03 EXPRESSION MATRICES ===
CGGA325_log2fpkm.csv.gz | 7.24 MB
CGGA693_log2fpkm.csv.gz | 9.7 MB
GSE4290_logexpr.csv.gz | 11.68 MB
GSE50161_logexpr.csv.gz | 4.63 MB
TCGA_logcpm.csv.gz | 21.16 MB
TOIL_counts.csv.gz | 50.35 MB
TOIL_logcpm.csv.gz | 60.85 MB


In [30]:
# NB03 READ-ONLY AUDIT. Paste into ONE Colab cell and run (Drive must be mounted).
# It modifies nothing in your project. Report is printed and saved to /content/nb03_audit_report.txt
import os, sys, re, gzip
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/GBM_MultiCohort_Research")
META = ROOT / "03_Metadata"
RAWD = ROOT / "02_Raw_Datasets"
OUT = []

def p(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    OUT.append(s)

def hdr(t):
    p("\n" + "=" * 8, t, "=" * 8)

# ---------------------------------------------------------------- 1. tree
hdr("1. Drive tree (files with sizes)")
if not ROOT.exists():
    p("ROOT MISSING:", ROOT)
else:
    for f in sorted(ROOT.rglob("*")):
        rel = f.relative_to(ROOT)
        if len(rel.parts) > 5 or any(x.startswith(".") for x in rel.parts):
            continue
        if f.is_dir():
            p("D", rel)
        else:
            p("F", rel, f"{f.stat().st_size / 1e6:.2f} MB")

# ---------------------------------------------------------------- 2. config + common
hdr("2. config.json")
cp = ROOT / "00_Project_Config" / "config.json"
p(cp.read_text(encoding="utf-8", errors="replace")[:8000] if cp.exists() else "MISSING config.json")

hdr("3. gbm_common.py definitions (names/signatures only)")
gp = ROOT / "00_Project_Config" / "gbm_common.py"
if gp.exists():
    for line in gp.read_text(encoding="utf-8", errors="replace").splitlines():
        if re.match(r"^(def |class |[A-Za-z_]+\s*=)", line):
            p(line)
else:
    p("MISSING gbm_common.py")

# ---------------------------------------------------------------- 4. NB02 sheets
hdr("4. NB02 metadata files")
names = ["TCGA_samples", "GTEx_samples", "CGGA325_samples", "CGGA693_samples",
         "GSE4290_samples", "GSE50161_samples",
         "GSE4290_label_mapping_audit", "GSE50161_label_mapping_audit",
         "cohort_flow", "exclusion_log", "dataset_inventory"]
sheets = {}
for n in names:
    f = META / f"{n}.csv"
    if not f.exists():
        p(f"-- {n}: MISSING")
        continue
    try:
        d = pd.read_csv(f, low_memory=False, encoding_errors="replace")
        sheets[n] = d
        p(f"-- {n}: shape={d.shape}")
        p("   columns:", list(d.columns))
        for c in d.columns:
            if len(d) > 12 and d[c].nunique(dropna=False) <= 12:
                p(f"   {c}: {d[c].value_counts(dropna=False).to_dict()}")
        p(d.head(3).to_string(max_colwidth=40))
        if len(d) <= 12:
            p(d.to_string(max_colwidth=40))
    except pd.errors.EmptyDataError:
        p(f"-- {n}: EMPTY (No columns to parse)")
    except Exception as e:
        p(f"-- {n}: ERROR loading file: {repr(e)}")

# ---------------------------------------------------------------- 5. GEO
hdr("5. GEO series matrices")
for acc in ["GSE4290", "GSE50161"]:
    f = RAWD / "GEO" / f"{acc}_series_matrix.txt.gz"
    p(f"-- {acc}: {f}")
    if not f.exists():
        p("   MISSING")
        continue
    wanted = ("!Series_platform_id", "!Sample_platform_id", "!Sample_data_processing",
              "!Sample_description", "!Sample_title", "!Sample_source_name_ch1")
    with gzip.open(f, "rt", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_begin"):
                break
            k = line.split("\t", 1)[0]
            if k in wanted or k.startswith("!Sample_characteristics"):
                vals = [v.strip().strip('"') for v in line.rstrip("\n").split("\t")[1:]]
                u = pd.Series(vals).value_counts()
                p(f"   {k}: n={len(vals)} unique={len(u)} top={str(u.head(4).to_dict())[:500]}")
    t = pd.read_csv(f, sep="\t", comment="!", index_col=0, low_memory=False, encoding_errors="replace")
    v = t.apply(pd.to_numeric, errors="coerce")
    arr = v.to_numpy(dtype="float64")
    q = np.nanpercentile(arr, [0, 1, 50, 99, 100])
    med = v.median()
    p(f"   table shape (rows x samples): {t.shape}; duplicated row IDs: {int(t.index.duplicated().sum())}")
    p("   first IDs:", list(t.index[:5]), "| first columns:", list(t.columns[:3]))
    p(f"   NaN fraction: {float(np.isnan(arr).mean()):.4f}")
    p("   value percentiles [0,1,50,99,100]:", np.round(q, 3).tolist())
    p(f"   per-sample median: min={med.min():.3f} max={med.max():.3f}")
    s = sheets.get(f"{acc}_label_mapping_audit")
    if s is not None:
        idc = [c for c in s.columns if c.lower() in ("sample_id", "geo_accession", "gsm", "sample")]
        if idc:
            ids = set(s[idc[0]].astype(str))
            p(f"   audit-file IDs in matrix columns: {len(ids & set(map(str, t.columns)))} of {len(ids)}")

# ---------------------------------------------------------------- 6. CGGA
hdr("6. CGGA files")
cg = RAWD / "CGGA"
if cg.exists():
    for f in sorted(cg.rglob("*")):
        if f.is_file() and not f.name.startswith("."):
            p("F", f.relative_to(ROOT), f"{f.stat().st_size / 1e6:.2f} MB")
    for f in sorted(cg.rglob("*")):
        if not f.is_file() or f.name.startswith(".") or f.suffix.lower() not in (".txt", ".tsv", ".csv"):
            continue
        nm = f.name.lower()
        sep = "," if f.suffix.lower() == ".csv" else "\t"
        if re.search(r"rsem|fpkm|expression", nm):
            e = pd.read_csv(f, sep=sep, index_col=0, low_memory=False, encoding_errors="replace")
            if not str(e.columns[0]).upper().startswith("CGGA") and str(e.index[0]).upper().startswith("CGGA"):
                e = e.T
                p("   (transposed for summary)")
            num = e.apply(pd.to_numeric, errors="coerce")
            cs = num.sum(axis=0)
            flat = num.to_numpy(dtype="float64").ravel()[:200000]
            flat = flat[~np.isnan(flat)]
            p(f"-- EXPR {f.name}: genes x samples = {e.shape}; dup gene IDs = {int(e.index.duplicated().sum())}")
            p("   first genes:", list(e.index[:3]), "| first samples:", list(e.columns[:3]))
            p(f"   min={np.nanmin(num.to_numpy()):.4g} max={np.nanmax(num.to_numpy()):.4g} NaN={int(num.isna().sum().sum())}")
            p(f"   column sums: min={cs.min():.4g} median={cs.median():.4g} max={cs.max():.4g}  (all ~1e6 suggests TPM)")
            p(f"   fraction integer-valued: {float((np.mod(flat, 1) == 0).mean()):.3f}  (~1 suggests counts)")
        elif re.search(r"clinical|meta|survival", nm):
            c = pd.read_csv(f, sep=sep, low_memory=False, encoding_errors="replace")
            p(f"-- CLIN {f.name}: shape={c.shape}")
            p("   columns:", list(c.columns))
            p(c.head(2).to_string(max_colwidth=30))
else:
    p("CGGA folder MISSING")

# ---------------------------------------------------------------- 7. Toil
hdr("7. Toil")
td = RAWD / "TOIL"
tf = td / "TcgaTargetGtex_gene_expected_count.gz"
if tf.exists():
    h = pd.read_csv(tf, sep="\t", compression="gzip", nrows=200, index_col=0, encoding_errors="replace")
    hv = h.to_numpy(dtype="float64")
    p(f"-- header: first column '{h.index.name}', {h.shape[1]} sample columns")
    p("   first gene IDs:", list(h.index[:3]))
    p(f"   value range (first 200 genes): {np.nanmin(hv):.3f} .. {np.nanmax(hv):.3f}")
    cols = pd.Series(h.columns.astype(str))
    p("   prefix counts:", cols.str.extract(r"^(TCGA|GTEX|TARGET)", expand=False).value_counts(dropna=False).to_dict())
    ids = set()
    for n in ("TCGA_samples", "GTEx_samples"):
        if n in sheets and "sample_id" in sheets[n]:
            ids |= set(sheets[n]["sample_id"].astype(str).str.strip())
    miss = sorted(ids - set(cols))
    p(f"   NB02 TCGA+GTEx sample IDs: {len(ids)}; absent from Toil header: {len(miss)} -> {miss[:10]}")
    for ph in td.rglob("*"):
        if ph.is_file() and not ph.name.startswith(".") and "phenotype" in ph.name.lower():
            d = pd.read_csv(ph, sep="\t", compression="infer", index_col=0, low_memory=False, encoding_errors="replace")
            sub = d.loc[d.index.isin(ids)]
            p(f"-- phenotype {ph.name}: {d.shape}; rows matching NB02 IDs: {len(sub)}")
            for c in d.columns:
                if c.lstrip("_") in ("study", "sample_type", "primary_site", "detailed_category"):
                    p(f"   {c}: {sub[c].value_counts(dropna=False).head(15).to_dict()}")
else:
    p("Toil expected_count file MISSING at", tf)
pm = td / "gencode.v23.annotation.gene.probemap"
p("probemap exists:", pm.exists())

Path("/content/nb03_audit_report.txt").write_text("\n".join(OUT), encoding="utf-8", errors="replace")
print("\nReport saved to /content/nb03_audit_report.txt -- copy the printed output (or this file) back to Claude.")


======== 1. Drive tree (files with sizes) ========
D 00_Project_Config
D 00_Project_Config/__pycache__
F 00_Project_Config/__pycache__/gbm_common.cpython-313.pyc 0.03 MB
F 00_Project_Config/config.json 0.00 MB
F 00_Project_Config/gbm_common.py 0.02 MB
D 00_Project_Config/r_scripts
F 00_Project_Config/r_scripts/map_probes.R 0.00 MB
F 00_Project_Config/r_scripts/nb03_map_probes.R 0.00 MB
F 00_Project_Config/r_scripts/r_install.R 0.00 MB
D 01_Literature_Review
D 02_Raw_Datasets
D 02_Raw_Datasets/CGGA
D 02_Raw_Datasets/CGGA/extracted_325
F 02_Raw_Datasets/CGGA/extracted_325/CGGA.mRNAseq_325.RSEM-genes.20200506.txt 40.03 MB
F 02_Raw_Datasets/CGGA/extracted_325/CGGA.mRNAseq_325_clinical.20200506.txt 0.03 MB
D 02_Raw_Datasets/CGGA/extracted_325/__MACOSX
D 02_Raw_Datasets/CGGA/extracted_693
F 02_Raw_Datasets/CGGA/extracted_693/CGGA.mRNAseq_693.RSEM-genes.20200506.txt 83.55 MB
F 02_Raw_Datasets/CGGA/extracted_693/CGGA.mRNAseq_693_clinical.20200506.txt 0.05 MB
D 02_Raw_Datasets/CGGA/extracted_6